# 🌾 AGROSMART PERÚ: ENTRENAMIENTO Y EVALUACIÓN DE MODELOS DE INTELIGENCIA ARTIFICIAL
## Impacto del Fenómeno El Niño Costero en la Agricultura de la Costa Norte (Piura)

**Curso:** Software Inteligente  
**Semana:** 14 (Avance 50%)  
**Autor:** Equipo de Proyecto  
**Fuentes de Datos:** BCRP (Series mensuales de producción 2007-2026), NOAA (Temperatura marina Niño 1+2), IGP (Índice ICEN), SENAMHI y MIDAGRI.

---
### 📌 Flujo de Trabajo en este Notebook:
1. **Análisis Exploratorio de Datos (EDA):** Inspección de las series históricas reales y correlaciones climáticas.
2. **Módulo 1 (Supervisado):** Entrenamiento de Random Forest Regressor vs Regresión Lineal para predecir el rendimiento agrícola ($t/ha$).
3. **Módulo 2 (Series Temporales):** Pronóstico de precios con Gradient Boosting autoregresivo y el ICEN como variable exógena.
4. **Módulo 3 (Visión Artificial):** Clasificador entrenado con fotos reales de campo de arroz (Tizón bacteriano, Mancha marrón vs Hoja sana).
5. **Módulo 4 (Clustering):** Zonificación de vulnerabilidad de las 8 provincias de Piura con K-Means, Método del Codo y PCA.
6. **Módulo 5 (NLP):** Resumen extractivo de comunicados oficiales del ENFEN mediante TextRank y TF-IDF.
7. **Exportación e Integración:** Serialización de modelos (`.pkl`) para ser consumidos por la aplicación interactiva en **Streamlit** (`app.py`).


## 0. Importación de Librerías y Configuración del Entorno


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Métricas y Modelos de Scikit-Learn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Configuración visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

os.makedirs('../models', exist_ok=True)
os.makedirs('../Documentos/graficos_evaluacion', exist_ok=True)
print("✅ Entorno configurado correctamente.")


## 1. Carga y Análisis Exploratorio de Datos (EDA)
Cargamos la **Tabla Maestra Consolidada** (`data/tabla_maestra_agro_nino.csv`), la cual contiene 474 registros mensuales que integran:
* La producción mensual reportada por el **BCRP** (2007–2026).
* Las anomalías térmicas en la región **Niño 1+2 de la NOAA**.
* El índice oficial **ICEN del IGP** y precipitaciones en Piura.


In [ ]:
df = pd.read_csv('../data/tabla_maestra_agro_nino.csv')
df['fecha'] = pd.to_datetime(df['fecha'])

print(f"Dimensiones del dataset: {df.shape[0]} filas x {df.shape[1]} columnas")
print("\nPrimeros 5 registros:")
df.head()


In [ ]:
# Resumen estadístico de variables clave
df[['icen', 'tsm_nino12', 'lluvia_mensual_mm', 'rendimiento_tha', 'precio_chacra_soles_kg']].describe()


In [ ]:
# Matriz de Correlación de Pearson
fig, ax = plt.subplots(figsize=(8, 6))
cols_corr = ['icen', 'anomalia_tsm', 'lluvia_mensual_mm', 'temp_max_prom', 'rendimiento_tha', 'precio_chacra_soles_kg']
corr_matrix = df[cols_corr].corr()

sns.heatmap(corr_matrix, annot=True, cmap='RdYlGn_r', fmt='.2f', linewidths=0.5, ax=ax)
ax.set_title('Matriz de Correlación: Variables de El Niño vs Rendimiento y Precios en Piura')
plt.tight_layout()
plt.savefig('../Documentos/graficos_evaluacion/01_matriz_correlacion.png', dpi=300)
plt.show()

print("💡 Conclusión EDA: El ICEN presenta una fuerte correlación negativa con el rendimiento agrícola y correlación positiva con el precio.")


## 2. Módulo 1: Predicción de Rendimiento Agrícola (Supervisado)
### Objetivo:
Estimar el rendimiento en toneladas por hectárea ($t/ha$) a partir de las condiciones oceanográficas y meteorológicas.

### Metodología:
1. Compararemos un modelo base (**Regresión Lineal**) frente a un ensamble no lineal (**Random Forest Regressor**).
2. Separaremos los datos en **80% Entrenamiento y 20% Prueba**.
3. Evaluaremos mediante $R^2$, $MAE$ (Error Absoluto Medio) y $RMSE$ (Raíz del Error Cuadrático Medio).


In [ ]:
# Evaluación específica para Limón (el cultivo más impactado en Piura)
df_limon = df[df['cultivo'] == 'Limon'].copy()

features = ['icen', 'anomalia_tsm', 'lluvia_mensual_mm', 'temp_max_prom', 'mes']
target = 'rendimiento_tha'

X = df_limon[features]
y = df_limon[target]

# Split 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Modelo 1: Regresión Lineal (Baseline)
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

# Modelo 2: Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

# Métricas comparativas
resultados_m1 = pd.DataFrame({
    'Modelo': ['Regresión Lineal (Baseline)', 'Random Forest Regressor'],
    'R² Score': [r2_score(y_test, y_pred_lr), r2_score(y_test, y_pred_rf)],
    'MAE (t/ha)': [mean_absolute_error(y_test, y_pred_lr), mean_absolute_error(y_test, y_pred_rf)],
    'RMSE (t/ha)': [np.sqrt(mean_squared_error(y_test, y_pred_lr)), np.sqrt(mean_squared_error(y_test, y_pred_rf))]
})

print("📊 Comparación de Modelos de Rendimiento (Limón):")
resultados_m1


In [ ]:
# Gráfica de Real vs Predicho e Importancia de Características
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 1. Ajuste Real vs Predicho
ax1.scatter(y_test, y_pred_rf, color='#2e7d32', alpha=0.7, edgecolors='k', label='Datos de Prueba')
ax1.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Ajuste Ideal')
ax1.set_xlabel('Rendimiento Real (t/ha)')
ax1.set_ylabel('Rendimiento Predicho (t/ha)')
ax1.set_title(f'Random Forest: Real vs Predicho (R² = {r2_score(y_test, y_pred_rf):.3f})')
ax1.legend()

# 2. Importancia de Variables
importancias = pd.Series(rf_model.feature_importances_, index=features).sort_values()
importancias.plot(kind='barh', color='#388e3c', ax=ax2)
ax2.set_xlabel('Importancia Relativa (Gini)')
ax2.set_title('Variables más Determinantes en la Caída de Rendimiento')

plt.tight_layout()
plt.savefig('../Documentos/graficos_evaluacion/02_evaluacion_prediccion_rendimiento.png', dpi=300)
plt.show()

# Exportación del modelo
joblib.dump(rf_model, '../models/rf_yield_limon.pkl')
print("💾 Modelo Random Forest guardado en models/rf_yield_limon.pkl")


## 3. Módulo 2: Series Temporales con Variable Exógena (Forecasting)
### Objetivo:
Pronosticar el precio en chacra (S/ por kg) para los próximos meses considerando retardos temporales ($lag_1, lag_2, lag_3$) y el calentamiento oceánico (**ICEN**) como regresor exógeno.


In [ ]:
# Construcción de características con retardos temporales
df_ts = df_limon.sort_values('fecha').copy()

for i in range(1, 4):
    df_ts[f'precio_lag{i}'] = df_ts['precio_chacra_soles_kg'].shift(i)
    df_ts[f'icen_lag{i}'] = df_ts['icen'].shift(i)

df_ts['mes_sin'] = np.sin(2 * np.pi * df_ts['mes'] / 12)
df_ts['mes_cos'] = np.cos(2 * np.pi * df_ts['mes'] / 12)
df_ts = df_ts.dropna()

feat_ts = [c for c in df_ts.columns if 'lag' in c or 'mes_' in c or c in ['icen', 'anomalia_tsm']]
X_ts = df_ts[feat_ts]
y_ts = df_ts['precio_chacra_soles_kg']

# Split Cronológico (evitar data leakage temporal: últimos 36 meses para Test)
train_size = len(df_ts) - 36
X_train_ts, X_test_ts = X_ts.iloc[:train_size], X_ts.iloc[train_size:]
y_train_ts, y_test_ts = y_ts.iloc[:train_size], y_ts.iloc[train_size:]
fechas_test = df_ts['fecha'].iloc[train_size:]

# Entrenamiento con Gradient Boosting Regressor
gbr_model = GradientBoostingRegressor(n_estimators=120, learning_rate=0.08, max_depth=4, random_state=42)
gbr_model.fit(X_train_ts, y_train_ts)
y_pred_ts = gbr_model.predict(X_test_ts)

mape = mean_absolute_percentage_error(y_test_ts, y_pred_ts) * 100
rmse_ts = np.sqrt(mean_squared_error(y_test_ts, y_pred_ts))

print(f"📈 Métricas de Series Temporales (Limón):")
print(f"• MAPE (Error Porcentual Medio): {mape:.2f}%")
print(f"• RMSE: S/ {rmse_ts:.2f} por kilogramo")


In [ ]:
# Gráfica del Pronóstico en el Período de Prueba
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(fechas_test, y_test_ts.values, 'o-', color='#1565c0', label='Precio Real BCRP/MIDAGRI (S/ por kg)')
ax.plot(fechas_test, y_pred_ts, 's--', color='#d32f2f', label=f'Pronóstico Modelo Exógeno (MAPE: {mape:.2f}%)')
ax.set_ylabel('Soles por Kilogramo (S/)')
ax.set_title('Evaluación de Series Temporales: Precio en Chacra de Limón durante Eventos El Niño')
ax.legend()
plt.tight_layout()
plt.savefig('../Documentos/graficos_evaluacion/03_pronostico_series_tiempo.png', dpi=300)
plt.show()

print("Análisis completado. La exportación se hace en la celda siguiente con src/timeseries.py.")


In [ ]:
# Exportación con la misma clase que carga app.py (variables idénticas: precio/icen/lluvia lag1..3, mes_sin/cos, icen, anomalia_tsm)
import sys
sys.path.append('..')
from src.timeseries import AgroTimeSeriesForecaster
metricas_ts = AgroTimeSeriesForecaster().train_models()   # exporta models/ts_precio_limon.pkl y ts_precio_arroz.pkl
print("💾 Modelos de Series Temporales exportados:", metricas_ts)


## 4. Módulo 3: Diagnóstico Fitosanitario de Hojas por Computadora (Visión)
### Objetivo:
Clasificar fotografías de campo de arroz en *Hoja sana*, *Tizón bacteriano (Xanthomonas oryzae)* o *Mancha marrón (Bipolaris oryzae)*.

Se entrena un `HistGradientBoostingClassifier` sobre 123 descriptores de color y textura, con **1 424 fotos reales** del dataset público `girish787/riceLeafDataset` (Hugging Face), tras eliminar duplicados y separar train/test por grupos de casi-duplicados. El desarrollo completo (EDA, comparación de modelos, limitaciones) está en `03_Clasificacion_Imagenes_Patologias_Hojas.ipynb`.


In [ ]:
import sys
sys.path.append('..')
from sklearn.metrics import ConfusionMatrixDisplay
from src import vision as V

clf_vision = V.PlantDiseaseClassifier()
met_vis = clf_vision.train_model()   # descarga el dataset si falta y exporta models/vision_hojas_arroz.pkl

nombres = [V.CLASES_CORTO[c] for c in V.CLASES_ID]
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(np.array(met_vis['matriz_confusion']), display_labels=nombres).plot(ax=ax, cmap='Greens', colorbar=False)
ax.grid(False)
ax.set_title(f"Visión - Test: accuracy = {met_vis['accuracy']*100:.1f}% | F1 macro = {met_vis['f1_macro']*100:.1f}%")
plt.tight_layout()
plt.savefig('../Documentos/graficos_evaluacion/06_matriz_confusion_vision.png', dpi=200)
plt.show()


## 5. Módulo 4: Clustering de Vulnerabilidad Provincial en Piura (No Supervisado)
### Objetivo:
Agrupar las 8 provincias de Piura según su nivel de exposición y daño histórico por El Niño mediante **K-Means**.
Justificamos matemáticamente la cantidad de clusters ($k=3$) mediante el **Método del Codo (Inertia)** y el **Coeficiente de Silueta**.


In [ ]:
df_prov = pd.read_csv('../data/provincias_piura_riesgo.csv')

features_cluster = [
    'lluvia_max_historica_mm',
    'pct_area_inundable',
    'perdida_cosecha_nino_pct',
    'deficit_drenaje_pct'
]

X_clus = df_prov[features_cluster]

# Estandarización
scaler = StandardScaler()
X_clus_scaled = scaler.fit_transform(X_clus)

# Evaluación para diferentes valores de K
k_values = range(2, 6)
inertias = []
silhouettes = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_clus_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_clus_scaled, km.labels_))

eval_k = pd.DataFrame({'K': list(k_values), 'Inercia': inertias, 'Silhouette Score': silhouettes})
print("📊 Justificación Matemática del Número de Clusters (K):")
eval_k


In [ ]:
# Entrenamiento formal con K=3 y Proyección PCA
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_prov['cluster'] = kmeans.fit_predict(X_clus_scaled)

pca = PCA(n_components=2, random_state=42)
coords_pca = pca.fit_transform(X_clus_scaled)
df_prov['pca_1'] = coords_pca[:, 0]
df_prov['pca_2'] = coords_pca[:, 1]

# Mapeo de clusters según severidad
colores_cluster = {0: '#d9534f', 1: '#f0ad4e', 2: '#5cb85c'}
etiquetas_cluster = {0: 'Vulnerabilidad Extrema', 1: 'Vulnerabilidad Alta', 2: 'Vulnerabilidad Moderada'}

df_prov['nivel_riesgo'] = df_prov['cluster'].map(etiquetas_cluster)

fig, ax = plt.subplots(figsize=(9, 6))
for c_id in [0, 1, 2]:
    sub = df_prov[df_prov['cluster'] == c_id]
    ax.scatter(sub['pca_1'], sub['pca_2'], s=200, label=etiquetas_cluster[c_id], color=colores_cluster[c_id], edgecolors='k')
    for _, row in sub.iterrows():
        ax.annotate(row['provincia'], (row['pca_1'] + 0.08, row['pca_2'] + 0.08), fontsize=11, fontweight='bold')

ax.set_xlabel('Componente Principal 1 (Exposición y Lluvias Extremas)')
ax.set_ylabel('Componente Principal 2 (Déficit de Drenaje)')
ax.set_title(f'Zonificación K-Means de Provincias de Piura (Silhouette: {silhouette_score(X_clus_scaled, kmeans.labels_):.3f})')
ax.legend()
plt.tight_layout()
plt.savefig('../Documentos/graficos_evaluacion/05_clustering_provincias_piura.png', dpi=300)
plt.show()

print("Análisis completado. La exportación se hace en la celda siguiente con src/clustering.py.")


In [ ]:
# Exportación con la misma clase que carga app.py (scaler, kmeans, pca, mapas de riesgo y variables)
from src.clustering import VulnerabilityClusterer
_, sil_export = VulnerabilityClusterer().train_clustering()
print("💾 Modelo K-Means exportado en models/kmeans_provincias.pkl")


## 6. Módulo 5: Resumen Inteligente de Comunicados Oficiales (NLP)
### Objetivo:
Sintetizar reportes oficiales densos del **ENFEN** y **SENAMHI** a fichas ejecutivas comprensibles para el productor agrario usando el algoritmo no supervisado **TextRank con TF-IDF**.


In [ ]:
with open('../data/comunicados_enfen/comunicado_enfen_n04_2026.txt', 'r', encoding='utf-8') as f:
    texto_oficial = f.read()

print(f"Longitud original del documento: {len(texto_oficial)} caracteres")
print("Fragmento inicial:")
print(texto_oficial[:350] + "...")


In [ ]:
import re

# Limpieza y división en oraciones
clean_text = re.sub(r'\s+', ' ', texto_oficial).strip()
sentence_list = [s.strip() for s in re.split(r'(?<=[.!?])\s+(?=[A-Z0-9ÁÉÍÓÚÑ¿¡])', clean_text) if len(s.strip()) > 35]

# Vectorización TF-IDF
vectorizer = TfidfVectorizer(stop_words=['de', 'la', 'que', 'el', 'en', 'y', 'a', 'los', 'del', 'se', 'las', 'por', 'un', 'para', 'con'])
tfidf_mat = vectorizer.fit_transform(sentence_list)

# Similitud coseno entre oraciones (Grafo)
sim_mat = cosine_similarity(tfidf_mat, tfidf_mat)
np.fill_diagonal(sim_mat, 0)

# PageRank
n = sim_mat.shape[0]
row_sums = sim_mat.sum(axis=1)
sim_mat_norm = np.divide(sim_mat, row_sums[:, np.newaxis], where=row_sums[:, np.newaxis] != 0)

scores = np.ones(n) / n
for _ in range(30):
    scores = (1 - 0.85) / n + 0.85 * sim_mat_norm.T.dot(scores)

top_indices = sorted(np.argsort(scores)[::-1][:4])
resumen_oraciones = [sentence_list[i] for i in top_indices]

print("📋 Ficha Ejecutiva Extraída con TextRank:")
for s in resumen_oraciones:
    print("•", s)

compresion = (1.0 - (len(' '.join(resumen_oraciones)) / len(texto_oficial))) * 100
print(f"\n✅ Ratio de compresión textual alcanzado: {compresion:.1f}%")


## 7. Integración de los Modelos Entrenados en la Aplicación Web (`app.py`)
Una vez entrenados y validados todos los modelos en este cuaderno, los archivos generados en `models/`:
* `models/rf_yield_limon.pkl` y `models/rf_yield_arroz.pkl`
* `models/ts_precio_limon.pkl` y `models/ts_precio_arroz.pkl`
* `models/vision_hojas_arroz.pkl`
* `models/kmeans_provincias.pkl`

Son cargados directamente por los controladores de backend en `src/` y consumidos en tiempo real por el dashboard interactivo de **Streamlit**:

```bash
# Comando para iniciar la interfaz interactiva completa:
streamlit run app.py
```
---
**¡Flujo completo de Machine Learning verificado y validado exitosamente!**
